# Project Friday: Unified Model Training Pipeline (Google Colab)

This notebook contains the complete pipeline to train and export the two primary local models for **Project Friday**:
1. **Wake-Word Detector (1D CNN M5)**: Trains a speaker-independent, noise-resilient 1D Convolutional Neural Network (M5 architecture) directly on raw PCM audio (`[1, 1, 24000]`) to detect "Friday". This avoids heavy on-device spectrogram calculations.
2. **NLU Intent Classifier (MobileBERT)**: Fine-tunes a lightweight MobileBERT model on a highly linguistically diverse command dataset, then quantizes it to dynamic INT8 ONNX.

### How to run this notebook in Google Colab:
1. Open [Google Colab](https://colab.research.google.com/).
2. Upload this `.ipynb` file (`scripts/friday_training.ipynb`).
3. Go to **Runtime -> Change runtime type** and select **T4 GPU** for acceleration.
4. Click **Runtime -> Run all**.
5. Download the final zipped package (`friday_onnx_models.zip`) containing `wakeword.onnx`, `nlu_model.onnx`, and `vocab.txt`, and copy them to your Android project's `app/src/main/assets/` directory.

## 1. Install Dependencies

In [ ]:
!pip install -q edge-tts transformers datasets onnx onnxruntime optimum torchaudio librosa soundfile nest_asyncio

## Step 1: Synthesize Positive Wake Word Samples using Edge-TTS
We cycle through multiple English neural voices (varying rate and pitch) to create 2,000+ diverse positive samples of the wake word "Friday".

In [ ]:
import os
import asyncio
import random
import glob
import nest_asyncio
import librosa
import soundfile as sf
import numpy as np
import edge_tts

os.makedirs("dataset/positive", exist_ok=True)
os.makedirs("dataset/negative", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Voices representing different genders, accents, and tones
VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-US-SteffanNeural",
    "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-IN-NeerjaNeural", "en-IN-PrabhatNeural",
    "en-AU-NatashaNeural", "en-AU-WilliamNeural", "en-CA-ClaraNeural", "en-CA-LiamNeural"
]

PHRASES = [
    "Friday", "Hey Friday", "Ok Friday", "Wake up Friday", "Friday assistant", 
    "Friday please", "listen Friday"
]

async def generate_single(voice, phrase, rate, pitch, output_file, sem):
    async with sem:
        try:
            communicate = edge_tts.Communicate(phrase, voice, rate=rate, pitch=pitch)
            await communicate.save(output_file)
            if os.path.exists(output_file) and os.path.getsize(output_file) > 1000:
                return True
        except Exception:
            pass
        return False

async def generate_positives(num_samples=2500):
    print(f"Synthesizing {num_samples} positive wake-word samples using edge-tts API (concurrently)...")
    sem = asyncio.Semaphore(40)  # Throttling to 40 concurrent requests to prevent rate-limiting
    tasks = []
    
    for i in range(num_samples):
        voice = random.choice(VOICES)
        phrase = random.choice(PHRASES)
        rate = f"{random.randint(-20, 20):+d}%"
        pitch = f"{random.randint(-10, 10):+d}Hz"
        output_file = f"dataset/positive/pos_{i}.mp3"
        tasks.append(generate_single(voice, phrase, rate, pitch, output_file, sem))
        
    results = await asyncio.gather(*tasks)
    successful = sum(1 for r in results if r)
    print(f"Done generating positive MP3s. Successfully generated {successful}/{num_samples} samples.")

nest_asyncio.apply()
asyncio.run(generate_positives(2000))

## Step 2: Download Negative Speech Commands & Noise Datasets
We download the Google Speech Commands v0.02 dataset to get a massive variety of short words (e.g. up, down, go, stop) to act as phonetic and conversational negatives, along with background noise profiles.

In [ ]:
print("Downloading Google Speech Commands dataset for negative samples...")
!wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz -O speech_commands.tar.gz
print("Extracting Speech Commands dataset...")
!mkdir -p dataset/google_speech
!tar -xzf speech_commands.tar.gz -C dataset/google_speech
print("Google Speech Commands downloaded and extracted successfully.")

## Step 3: Audio Preprocessing & Augmentation
We convert synthesized positive MP3 files to 16kHz WAV, align them to a fixed 1.5s duration, and implement robust training-time augmentations.

In [ ]:
SAMPLE_RATE = 16000
DURATION_SECONDS = 1.5
INPUT_SIZE = int(SAMPLE_RATE * DURATION_SECONDS)  # 24000 samples

# 1. Convert MP3s to 16kHz WAV in parallel
import concurrent.futures
pos_mp3s = glob.glob("dataset/positive/*.mp3")
print(f"Converting {len(pos_mp3s)} MP3 files to 16kHz WAV in parallel...")

def convert_single(f):
    try:
        y, sr = librosa.load(f, sr=SAMPLE_RATE)
        wav_path = f.replace(".mp3", ".wav")
        sf.write(wav_path, y, SAMPLE_RATE)
        os.remove(f)
    except Exception:
        pass

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    executor.map(convert_single, pos_mp3s)

# 2. Load background noise files
noise_files = glob.glob("dataset/google_speech/_background_noise_/*.wav")
noise_profiles = []
for f in noise_files:
    if "README" in f: continue
    y, sr = librosa.load(f, sr=SAMPLE_RATE)
    noise_profiles.append(y)
print(f"Loaded {len(noise_profiles)} background noise loops.")

# 3. Raw PCM processing helper
def preprocess_raw_audio(y, target_len=INPUT_SIZE):
    if len(y) > target_len:
        start = (len(y) - target_len) // 2
        y = y[start:start + target_len]
    else:
        pad_len = target_len - len(y)
        left = pad_len // 2
        right = pad_len - left
        y = np.pad(y, (left, right), 'constant')
    return y.astype(np.float32)

# 4. Data Augmentation
def augment_audio(y, noise_profiles):
    # Random Shift (up to 150ms)
    shift = np.random.randint(-2400, 2400)
    if shift > 0:
        y_aug = np.pad(y, (shift, 0), 'constant')[:-shift]
    elif shift < 0:
        y_aug = np.pad(y, (0, -shift), 'constant')[-shift:]
    else:
        y_aug = y.copy()
    
    # Random Gain
    y_aug = y_aug * np.random.uniform(0.6, 1.4)
    
    # Add environmental noise
    if len(noise_profiles) > 0 and np.random.rand() > 0.3:
        noise = random.choice(noise_profiles)
        start_idx = np.random.randint(0, len(noise) - len(y_aug))
        noise_slice = noise[start_idx:start_idx + len(y_aug)]
        snr = np.random.uniform(5, 18)
        y_rms = np.sqrt(np.mean(y_aug**2) + 1e-8)
        n_rms = np.sqrt(np.mean(noise_slice**2) + 1e-8)
        scaled_noise = noise_slice * (y_rms / (n_rms * (10**(snr/20))))
        y_aug = y_aug + scaled_noise
        
    return np.clip(y_aug, -1.0, 1.0)

## Step 4: Build Dataset Matrices
We load positive wavs and sample 5,000+ negative wavs from Google Speech Commands (including words like *yes, no, up, down, stop, go, zero, one*), and perform augmentations.

In [ ]:
import concurrent.futures
X = []
y = []

# 1. Load Positives & Augment in parallel
pos_files = glob.glob("dataset/positive/*.wav")
print(f"Processing {len(pos_files)} positive files in parallel...")

def process_single_pos(f):
    try:
        wav, _ = librosa.load(f, sr=SAMPLE_RATE)
        processed = preprocess_raw_audio(wav)
        aug = preprocess_raw_audio(augment_audio(processed, noise_profiles))
        return (processed, aug)
    except Exception:
        return None

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(process_single_pos, pos_files))

for r in results:
    if r is not None:
        X.append(r[0])
        y.append(1)
        X.append(r[1])
        y.append(1)

# 2. Gather Negatives
neg_words = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go", "zero", "one", "two", "three"]
neg_files = []
for word in neg_words:
    word_dir = os.path.join("dataset/google_speech", word)
    if os.path.exists(word_dir):
        neg_files.extend(glob.glob(word_dir + "/*.wav")[:400])

print(f"Processing {len(neg_files)} negative files in parallel...")

def process_single_neg(f):
    try:
        wav, _ = librosa.load(f, sr=SAMPLE_RATE)
        processed = preprocess_raw_audio(wav)
        aug = preprocess_raw_audio(augment_audio(processed, noise_profiles))
        return (processed, aug)
    except Exception:
        return None

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(process_single_neg, neg_files))

for r in results:
    if r is not None:
        X.append(r[0])
        y.append(0)
        X.append(r[1])
        y.append(0)

# 3. Add noise/silence negatives
for _ in range(300):
    # Pure silence
    X.append(np.zeros(INPUT_SIZE, dtype=np.float32))
    y.append(0)
    # Ambient noise only
    if len(noise_profiles) > 0:
        noise = random.choice(noise_profiles)
        start_idx = np.random.randint(0, len(noise) - INPUT_SIZE)
        X.append(preprocess_raw_audio(noise[start_idx:start_idx + INPUT_SIZE]))
        y.append(0)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

# Add channel dimension for 1D-CNN: (Batch, 1, 24000)
X = np.expand_dims(X, axis=1)

print(f"Final Dataset Shape: {X.shape}")
print(f"Positives: {np.sum(y == 1)}, Negatives: {np.sum(y == 0)}")

## Step 5: Define 1D CNN M5 Wake-Word Model & Train
We use the classic M5 architecture, which applies deep 1D convolutions directly to raw PCM audio. It has ~500k parameters and performs exceptionally well on keyword spotting.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class M5(nn.Module):
    def __init__(self, n_input=1, n_output=2, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = nn.BatchNorm1d(n_channel)
        self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(n_channel)
        self.pool2 = nn.MaxPool1d(4)
        self.conv3 = nn.Conv1d(n_channel, 2 * n_channel, kernel_size=3)
        self.bn3 = nn.BatchNorm1d(2 * n_channel)
        self.pool3 = nn.MaxPool1d(4)
        self.conv4 = nn.Conv1d(2 * n_channel, 2 * n_channel, kernel_size=3)
        self.bn4 = nn.BatchNorm1d(2 * n_channel)
        self.pool4 = nn.MaxPool1d(4)
        self.fc1 = nn.Linear(2 * n_channel, n_output)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.conv2(x)
        x = F.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.conv3(x)
        x = F.relu(self.bn3(x))
        x = self.pool3(x)
        x = self.conv4(x)
        x = F.relu(self.bn4(x))
        x = self.pool4(x)
        x = F.adaptive_avg_pool1d(x, 1)
        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        return x.squeeze(1)

# Split dataset (90% Train, 10% Val)
indices = np.arange(len(X))
np.random.shuffle(indices)
X, y = X[indices], y[indices]

split = int(len(X) * 0.9)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = M5().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train loop
epochs = 20
print(f"Training 1D CNN M5 Wake Word Model on {device}...")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == batch_y).item()
        
    train_loss = total_loss / len(X_train)
    train_acc = correct / len(X_train)
    
    # Val
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_x.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == batch_y).item()
            
    val_loss = val_loss / len(X_val)
    val_acc = val_correct / len(X_val)
    
    print(f"Epoch {epoch}/{epochs} - Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

## Step 6: Export Wake-Word Model to ONNX
We export the PyTorch model to ONNX, force IR version 8 (to guarantee compatibility with ONNX Runtime on Android), and package the weights natively inside the ONNX file.

In [ ]:
import onnx
from onnx.external_data_helper import convert_model_from_external_data

onnx_path = "output/wakeword.onnx"
print(f"Exporting wake word model to ONNX: {onnx_path}...")

model.eval().to("cpu")
dummy_input = torch.randn(1, 1, INPUT_SIZE, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input_audio"],
    output_names=["probabilities"],
    opset_version=15
)

# Fix IR version compatibility for Android ONNX Runtime
model_proto = onnx.load(onnx_path)
model_proto.ir_version = 8
convert_model_from_external_data(model_proto)
onnx.save_model(model_proto, onnx_path)

print("[+] Wake-word ONNX model successfully finalized.")

## Step 7: Synthesize NLU Command Dataset
To handle linguistic variations (politeness, indirect commands, active/passive), we generate a massive, diverse command-classification dataset containing 10,000+ total phrases covering 16 distinct assistant intents.

In [ ]:
import json
import pandas as pd

INTENT_LABELS = [
    "volume_up", "volume_down", "brightness_up", "brightness_down", 
    "torch_toggle", "torch_strength", "lock_phone", "open_app", 
    "navigate_to", "set_alarm", "set_timer", "send_whatsapp", 
    "play_media", "power_saver_toggle", "screencast_toggle", "unknown"
]

polite_prefixes = [
    "", "please ", "can you ", "could you ", "would you mind ", 
    "would you please ", "hey friday ", "friday please ", "will you ", 
    "can you please ", "could you please ", "i want to ", "assistant please "
]

verbs = {
    "volume_up": ["increase", "raise", "turn up", "crank up", "boost", "make louder", "turn up sound in"],
    "volume_down": ["decrease", "lower", "turn down", "make quieter", "reduce", "make softer", "hush"],
    "brightness_up": ["increase", "raise", "turn up", "boost", "make brighter", "brighten"],
    "brightness_down": ["decrease", "lower", "turn down", "dim", "make darker", "reduce"],
    "torch_toggle": ["turn on", "turn off", "switch on", "switch off", "toggle", "enable", "disable", "activate", "kill"],
    "torch_strength": ["set flashlight strength to", "change torch level to", "set torch intensity to", "make torch level"],
    "lock_phone": ["lock", "turn off", "put to sleep", "shutdown"],
    "open_app": ["open", "launch", "start", "run", "go to", "show me"],
    "navigate_to": ["navigate to", "show directions to", "go to", "find routes to", "how do i get to"],
    "set_alarm": ["set an alarm for", "wake me up at", "create an alarm for", "alarm for"],
    "set_timer": ["set a timer for", "start a timer for", "countdown for"],
    "send_whatsapp": ["send a whatsapp to", "whatsapp", "message", "text", "ping"],
    "play_media": ["play", "start playing", "resume", "listen to"],
    "power_saver_toggle": ["turn on", "turn off", "enable", "disable", "toggle", "activate"],
    "screencast_toggle": ["turn on", "turn off", "enable", "disable", "toggle", "start", "stop"]
}

targets = {
    "volume_up": ["volume", "sound", "audio", "speaker", "music volume"],
    "volume_down": ["volume", "sound", "audio", "speaker", "music volume"],
    "brightness_up": ["brightness", "screen brightness", "display brightness", "screen illumination"],
    "brightness_down": ["brightness", "screen brightness", "display brightness", "screen"],
    "torch_toggle": ["flashlight", "torch", "flash", "light"],
    "lock_phone": ["phone", "screen", "display", "device"],
    "power_saver_toggle": ["power saver", "battery saver", "eco mode", "low power mode"],
    "screencast_toggle": ["screen cast", "screencasting", "smart view", "mirroring"]
}

suffixes = ["", " a bit", " a little", " please", " right now", " immediately", " some more"]

nlu_data = []

# Programmatic expansion of core intents
for intent, verb_list in verbs.items():
    for v in verb_list:
        for p in polite_prefixes:
            for s in suffixes:
                if intent in targets:
                    for t in targets[intent]:
                        nlu_data.append((f"{p}{v} {t}{s}".strip(), intent))
                else:
                    # Intents requiring arguments
                    if intent == "open_app":
                        for app in ["whatsapp", "spotify", "instagram", "youtube", "gmail", "settings", "browser", "maps"]:
                            nlu_data.append((f"{p}{v} {app}{s}".strip(), intent))
                    elif intent == "navigate_to":
                        for loc in ["new york", "london", "home", "work", "starbucks", "the nearest hospital", "grocery store"]:
                            nlu_data.append((f"{p}{v} {loc}{s}".strip(), intent))
                    elif intent == "set_alarm":
                        for time in ["7 am", "8:30 pm", "six in the morning", "noon", "midnight"]:
                            nlu_data.append((f"{p}{v} {time}{s}".strip(), intent))
                    elif intent == "set_timer":
                        for duration in ["5 minutes", "10 mins", "one hour", "30 seconds"]:
                            nlu_data.append((f"{p}{v} {duration}{s}".strip(), intent))
                    elif intent == "send_whatsapp":
                        for contact in ["mom", "dad", "brother", "boss", "john", "mary"]:
                            nlu_data.append((f"{p}{v} {contact} saying hello{s}".strip(), intent))
                            nlu_data.append((f"{p}{v} {contact}{s}".strip(), intent))
                    elif intent == "play_media":
                        for media in ["lofi hip hop", "some rock music", "podcast", "taylor swift", "jazz"]:
                            nlu_data.append((f"{p}{v} {media}{s}".strip(), intent))
                    elif intent == "torch_strength":
                        for strength in ["five", "3", "maximum", "level 2", "full strength"]:
                            nlu_data.append((f"{p}{v} {strength}{s}".strip(), intent))

# Add indirect phrasing & slang expressions
indirect_phrasings = [
    # Volume
    ("it is too quiet in here", "volume_up"),
    ("i cannot hear anything", "volume_up"),
    ("crank up the audio", "volume_up"),
    ("it is extremely loud", "volume_down"),
    ("my ears are hurting", "volume_down"),
    ("pipe down the noise", "volume_down"),
    ("quiet down", "volume_down"),
    # Brightness
    ("the screen is too dim", "brightness_up"),
    ("i cannot see the screen", "brightness_up"),
    ("make the screen brighter", "brightness_up"),
    ("it is too bright", "brightness_down"),
    ("the display is blinding me", "brightness_down"),
    ("dim down the screen", "brightness_down"),
    # Torch
    ("it is pitch dark in here", "torch_toggle"),
    ("we need some light", "torch_toggle"),
    ("kill the torch", "torch_toggle"),
    ("extinguish the flashlight", "torch_toggle"),
    # Unknowns (Chat Fallbacks)
    ("what is the capital of france", "unknown"),
    ("who wrote hamlet", "unknown"),
    ("tell me a funny joke", "unknown"),
    ("how is the weather in tokio", "unknown"),
    ("define photosynthesis", "unknown"),
    ("hello how are you today", "unknown"),
    ("what is your name", "unknown"),
    ("recommend a good book", "unknown"),
    ("who is the president of the US", "unknown"),
    ("can pigs fly", "unknown")
]

for phrase, intent in indirect_phrasings:
    nlu_data.append((phrase, intent))

# Convert to DataFrame and drop duplicates
df_nlu = pd.DataFrame(nlu_data, columns=["text", "label"])
df_nlu = df_nlu.drop_duplicates().sample(frac=1.0).reset_index(drop=True)

print(f"Total unique NLU training sentences generated: {len(df_nlu)}")
print(df_nlu["label"].value_counts())

## Step 8: Fine-Tune MobileBERT Intent Classifier
We load pre-trained `google/mobilebert-uncased`, map the intent labels, tokenize the dataset, and run HuggingFace sequence classification training.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

model_name = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Save vocabulary file for Android WordPiece Tokenizer integration
vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
vocab_path = "output/vocab.txt"
with open(vocab_path, "w", encoding="utf-8") as f:
    for word, _ in sorted_vocab:
        f.write(word + "\n")
print(f"[+] Saved tokenizer vocabulary to {vocab_path}")

# Encode intent labels to indices
label_to_id = {label: i for i, label in enumerate(INTENT_LABELS)}
id_to_label = {i: label for i, label in enumerate(INTENT_LABELS)}

class NluDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.encodings = tokenizer(texts, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = [label_to_id[l] for l in labels]
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Split dataset (90% Train, 10% Val)
train_split = df_nlu.sample(frac=0.9, random_state=42)
val_split = df_nlu.drop(train_split.index)

train_dataset = NluDataset(train_split["text"].tolist(), train_split["label"].tolist(), tokenizer)
val_dataset = NluDataset(val_split["text"].tolist(), val_split["label"].tolist(), tokenizer)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(INTENT_LABELS))

training_args = TrainingArguments(
    output_dir="./nlu_results",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=50,
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Starting MobileBERT fine-tuning...")
trainer.train()
print("[+] Training finished!")

## Step 9: Export and Quantize NLU Model
We export the fine-tuned model to ONNX, apply Dynamic INT8 Quantization (reducing model size from 100MB to ~24MB), and finalize it with Android-compatible settings.

In [ ]:
import os
import shutil
import onnx
from optimum.exporters.onnx import main_export
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnx.external_data_helper import convert_model_from_external_data

# 1. Save PyTorch model and tokenizer locally
print("Saving fine-tuned PyTorch model and tokenizer...")
model.save_pretrained("./nlu_model_dir")
tokenizer.save_pretrained("./nlu_model_dir")

# 2. Export to ONNX using HF Optimum
print("Exporting MobileBERT NLU model to ONNX using Hugging Face Optimum...")
main_export(
    model_name_or_path="./nlu_model_dir",
    output="output/",
    task="text-classification"
)

nlu_onnx_path = "output/model.onnx"
final_quant_path = "output/nlu_model.onnx"

# 3. Perform Dynamic INT8 Quantization
print("Applying dynamic INT8 quantization to NLU ONNX...")
quantize_dynamic(
    model_input=nlu_onnx_path,
    model_output=final_quant_path,
    weight_type=QuantType.QInt8
)

# 4. Downgrade IR version to 8 for Android compatibility
print("Downgrading ONNX model IR version to 8 for Android compatibility...")
model_proto = onnx.load(final_quant_path)
model_proto.ir_version = 8
convert_model_from_external_data(model_proto)
onnx.save_model(model_proto, final_quant_path)

# 5. Clean up raw non-quantized model and configuration files
print("Cleaning up temporary files...")
for f in ["model.onnx", "config.json", "special_tokens_map.json", "tokenizer_config.json", "tokenizer.json"]:
    path = os.path.join("output", f)
    if os.path.exists(path):
        os.remove(path)

if os.path.exists("./nlu_model_dir"):
    shutil.rmtree("./nlu_model_dir")

print("[+] Quantized NLU ONNX model successfully finalized.")

## Step 10: Zip Output Files for Download
We bundle all the final compiled files (`wakeword.onnx`, `nlu_model.onnx`, `vocab.txt`) into a single zip file ready to be copied into Friday's assets.

In [ ]:
!zip -j output/friday_onnx_models.zip output/wakeword.onnx output/nlu_model.onnx output/vocab.txt
print("\n===============================================")
print("=== TRAINING COMPLETED SUCCESSFULLY ===")
print("===============================================")
print("Final artifacts zipped at: output/friday_onnx_models.zip")
print("Download this zip and extract its contents into your app assets directory: app/src/main/assets/")